# M3 — Detección de ejes vehiculares | V2
**Sistema de control de peaje · Google Colab · YOLOv8 Pose**

Entrenamiento del dataset combinado `m3_adaptado_v2` (Vehicle Wheel Pose + Truck Image Dataset v2.0.0). Clase única `vehiculo`, siete posiciones semánticas del frente hacia atrás. Cada fila YOLO contiene 26 valores: clase + caja + 7 tripletas (x, y, visibilidad).

**Antes de empezar:** sube `m3_adaptado_v2.zip` a `Mi unidad/Sistema-Peaje/datasets/`. Debe incluir `train`, `valid`, `test`, `data.yaml` y `metadatos_procedencia.json` (el reporte de conversión es recomendable). Activa GPU T4 en Colab. Ejecuta las secciones en orden; no necesitas Roboflow ni convertir COCO otra vez.

Referencias del reporte v2: 1031/318/164 imágenes y 2738/612/269 instancias. Hay 611 grupos declarados de procedencia y 0 cruces reportados. No se garantiza independencia de escenas no identificadas. Los conteos de referencia provienen de las anotaciones, no de una inspección manual independiente. No incluye evidencia de imágenes propias nuevas: incorpóralas según la consigna del laboratorio.

**Alcance:** hasta siete ejes; las ruedas ocultas, las orientaciones dudosas y los vehículos de más de siete ejes requieren revisión. Un punto detectado no equivale automáticamente a un eje real confirmado ni autoriza un cobro.

## 1. Instalar dependencias y comprobar GPU

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9', 'pyyaml'])
import torch, ultralytics
from ultralytics import YOLO
assert torch.cuda.is_available(), 'Activa GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)

## 2. Conectar Drive y configurar
Para una prueba rápida puedes usar otro `RUN_NAME` y `EPOCHS=1`. Para reanudar una ejecución interrumpida, conserva los parámetros y cambia `RESUME=True`; no lo hagas sobre una ejecución terminada. Si falta memoria, usa BATCH=4 con un nuevo nombre de ejecución. El umbral inicial de keypoints es 0.50: no se debe ajustar usando test.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, yaml, shutil, os, math, random, hashlib, re, zipfile
from collections import Counter, defaultdict
import numpy as np
from PIL import Image as PILImage, ImageDraw

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Sistema-Peaje')
for folder in ['datasets','modelos','entrenamientos','evidencias']:
    (BASE/folder).mkdir(parents=True, exist_ok=True)
ZIP = BASE/'datasets'/'m3_adaptado_v2.zip'
RUN_NAME = 'M3_keypoints_v2'
EPOCHS = 100
PATIENCE = 20
ALLOW_EXCLUSIONS = False  # Solo True tras revisar el reporte de exclusiones.
IMGSZ = 640   # Si los objetos son demasiado pequeños, prueba 960 en otra ejecución.
BATCH = 8     # Si falta VRAM, baja a 4 o 2 antes de iniciar una nueva ejecución.
SEED = 42
RESUME = False
KPT_CONF = 0.50
BOX_CONF = 0.25
MAX_DET = 300
RUN = BASE/'entrenamientos'/RUN_NAME
EVID = BASE/'evidencias'/RUN_NAME
EVID.mkdir(parents=True, exist_ok=True)
assert ZIP.is_file(), f'Sube el ZIP YOLO Pose a Drive: {ZIP}'
print('Dataset:', ZIP, '| Resultados:', RUN)

## 3. Extraer el ZIP y cargar metadatos
Se acepta el ZIP con o sin carpeta superior. Se trabaja en almacenamiento temporal; el ZIP de Drive no se modifica.

In [ ]:
def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
ZIP_HASH = sha_file(ZIP)
EXTRACT = Path('/content')/('peaje_m3_v2_'+ZIP_HASH[:12])
MARK = EXTRACT/'.extraction_complete'
if not MARK.exists():
    if EXTRACT.exists(): shutil.rmtree(EXTRACT)
    EXTRACT.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as z:
        for member in z.infolist():
            target = (EXTRACT/member.filename).resolve()
            if not target.is_relative_to(EXTRACT.resolve()):
                raise ValueError('Ruta insegura dentro del ZIP')
        z.extractall(EXTRACT)
    MARK.touch()
candidates = [p for p in EXTRACT.rglob('data.yaml') if (p.parent/'train'/'images').is_dir()]
assert len(candidates)==1, 'No se encontró una única raíz YOLO Pose. Comprueba que subiste m3_adaptado_v2 y no coco-m3.'
ROOT = candidates[0].parent
CFG = yaml.safe_load(candidates[0].read_text(encoding='utf-8-sig'))
print('Raíz:', ROOT)
print(yaml.safe_dump(CFG, sort_keys=False))
META_FILE=ROOT/'metadatos_procedencia.json'
assert META_FILE.is_file(), 'Falta metadatos_procedencia.json junto a data.yaml. Revisa el ZIP.'
META=json.loads(META_FILE.read_text(encoding='utf-8-sig'))
assert META['esquema']['clase']=='vehiculo' and META['esquema']['kpt_shape']==[7,3]
assert META['esquema']['keypoints']==[f'eje_{i}' for i in range(1,8)]
META_INDEX={}
for rec in META['imagenes_incluidas']:
    key=(rec['split'],rec['archivo_imagen'])
    assert key not in META_INDEX, f'Metadato duplicado: {key}'
    META_INDEX[key]=rec
print('Metadatos:', len(META_INDEX), 'imágenes')
conversion=ROOT/'reporte_conversion_v2.json'
if conversion.is_file():
    shutil.copy2(conversion,EVID/conversion.name)
shutil.copy2(META_FILE,EVID/META_FILE.name)

## 4. Validar etiquetas, imágenes y correspondencia con metadatos
Cada fila se coteja con el registro de procedencia en el mismo orden en que lo exportó el conversor. Los puntos de visibilidad 0 deben ser (0,0,0). Los valores 1 y 2 indican puntos anotados; 1 puede representar oclusión. Se revisa el archivo de imagen y se genera una copia de trabajo independiente. Una imagen inválida completa se excluye de la copia (no del ZIP); por defecto el notebook se detiene si hay exclusiones. No elimina automáticamente anotaciones válidas marcadas para revisión: revisa las muestras de la sección 6. No se eliminan puntos individuales, evitando vehículos sin etiquetar.

In [ ]:
names = CFG.get('names')
names = {int(k):str(v) for k,v in names.items()} if isinstance(names,dict) else dict(enumerate(names or []))
assert names == {0:'vehiculo'}, f'Clase incorrecta: {names}'
assert list(CFG.get('kpt_shape',[])) == [7,3], 'Se requiere kpt_shape [7,3]'
EXTS={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
SPLITS={s:ROOT/s/'images' for s in ('train','valid','test')}
if not SPLITS['valid'].is_dir() and (ROOT/'val'/'images').is_dir(): SPLITS['valid']=ROOT/'val'/'images'
KEYPOINT_NAMES=[f'eje_{i}' for i in range(1,8)]

def read_pose(label):
    rows=[]
    for lineno,line in enumerate(label.read_text(encoding='utf-8-sig').splitlines(),1):
        if not line.strip(): continue
        v=[float(x) for x in line.split()]
        if len(v)!=26 or not all(math.isfinite(x) for x in v): raise ValueError(f'línea {lineno}: longitud o valor inválido')
        if v[0]!=0 or not all(0<=x<=1 for x in v[1:5]) or min(v[3:5])<=0: raise ValueError(f'línea {lineno}: clase o bbox inválido')
        x,y,w,h=v[1:5]
        if min(x-w/2,y-h/2)<-1e-5 or max(x+w/2,y+h/2)>1+1e-5: raise ValueError(f'línea {lineno}: bbox fuera de imagen')
        k=np.array(v[5:]).reshape(7,3)
        if not np.isin(k[:,2],[0,1,2]).all() or not ((k[:,:2]>=0)&(k[:,:2]<=1)).all(): raise ValueError(f'línea {lineno}: keypoints inválidos')
        if np.any(k[k[:,2]==0,:2]!=0) or not np.any(k[:,2]>0): raise ValueError(f'línea {lineno}: visibilidad incoherente')
        rows.append((v[1:5],k))
    return rows


CLEAN=Path('/content')/('m3_v2_limpio_'+ZIP_HASH[:12])
if CLEAN.exists(): shutil.rmtree(CLEAN)
IMAGES={}; REPORT={}; REJECTED=[]; CLEAN_META={}; GT={}; SEEN=set()
for split,idir in SPLITS.items():
    ldir=idir.parent/'labels'
    assert idir.is_dir() and ldir.is_dir(), f'Falta images o labels en {split}'
    (CLEAN/split/'images').mkdir(parents=True)
    (CLEAN/split/'labels').mkdir(parents=True)
    imgs=sorted(p for p in idir.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
    assert imgs, f'Conjunto vacío: {split}'
    real=Counter(); annotated=Counter(); visible=Counter(); support=Counter(); sources=Counter(); flagged=0; kept=[]; expected_labels=set()
    for im in imgs:
        rel=im.relative_to(idir); label=ldir/rel.with_suffix('.txt'); expected_labels.add(label)
        key=(split,im.name); rec=META_INDEX.get(key)
        try:
            if key in SEEN: raise ValueError('nombre de imagen duplicado en split')
            SEEN.add(key)
            if rec is None: raise ValueError('imagen sin metadatos de procedencia')
            if rec['archivo_etiqueta']!=label.name: raise ValueError('nombre de etiqueta no coincide')
            if not label.is_file(): raise ValueError('sin TXT')
            rows=read_pose(label)
            if not rows: raise ValueError('TXT vacío no verificado')
            if len(rows)!=len(rec['instancias']): raise ValueError('número de filas distinto de metadatos')
            for (_,k),m in zip(rows,rec['instancias']):
                nr=m['ejes_reales']; na=m['ejes_visibles']; actual=int((k[:,2]>0).sum())
                if not isinstance(nr,int) or not isinstance(na,int) or not 1<=actual==na<=nr<=7:
                    raise ValueError('conteo de puntos no coincide con metadatos')
                if np.any(k[nr:,2]>0): raise ValueError('punto anotado más allá del número de ejes')
            with PILImage.open(im) as pic: pic.verify()
            with PILImage.open(im) as pic: pic.load()
        except (ValueError,OSError,KeyError,TypeError) as e:
            REJECTED.append({'split':split,'imagen':str(rel),'motivo':str(e)}); continue
        dest=CLEAN/split/'images'/rel; dest.parent.mkdir(parents=True,exist_ok=True)
        dstlabel=CLEAN/split/'labels'/rel.with_suffix('.txt'); dstlabel.parent.mkdir(parents=True,exist_ok=True)
        try: os.link(im,dest)
        except OSError: shutil.copy2(im,dest)
        shutil.copy2(label,dstlabel); kept.append(dest); CLEAN_META[dest]=rec; GT[dest]=rows
        for (_,k),m in zip(rows,rec['instancias']):
            real[m['ejes_reales']]+=1; annotated[int((k[:,2]>0).sum())]+=1; visible[int((k[:,2]==2).sum())]+=1
            support.update(j+1 for j,v in enumerate(k[:,2]) if v>0)
            flagged+=int(m['requiere_revision']); sources[rec['fuente']]+=1
    orphan=set(ldir.rglob('*.txt'))-expected_labels
    IMAGES[split]=kept
    REPORT[split]={'originales':len(imgs),'utilizadas':len(kept),'instancias':sum(real.values()),'ejes_reales_referencia':dict(sorted(real.items())), 'puntos_anotados_v_mayor_0':dict(sorted(annotated.items())), 'puntos_visibles_v_2':dict(sorted(visible.items())), 'soporte_por_keypoint':dict(sorted(support.items())), 'marcadas_revision':flagged,'instancias_por_fuente':dict(sources),'etiquetas_huerfanas':len(orphan)}
MISSING_META=sorted(set(META_INDEX)-SEEN)
validation={'resumen':REPORT,'excluidas':REJECTED,'metadatos_sin_imagen':MISSING_META,'zip_sha256':ZIP_HASH}
(EVID/'validacion_dataset.json').write_text(json.dumps(validation,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(REPORT,indent=2,ensure_ascii=False)); print('Excluidas:',len(REJECTED),'| Metadatos sin imagen:',len(MISSING_META))
for r in REJECTED[:15]: print(r)
assert all(IMAGES.values()), 'Hay un conjunto vacío después de validar.'
assert ALLOW_EXCLUSIONS or not (REJECTED or MISSING_META or any(r['etiquetas_huerfanas'] for r in REPORT.values())), 'Revisa validacion_dataset.json; corrige el ZIP o habilita ALLOW_EXCLUSIONS explícitamente.'
DATA=CLEAN/'data.yaml'
DATA.write_text(yaml.safe_dump({'path':str(CLEAN),'train':'train/images','val':'valid/images','test':'test/images','names':{0:'vehiculo'},'nc':1,'kpt_shape':[7,3],'flip_idx':list(range(7)),'kpt_names':{0:KEYPOINT_NAMES}},sort_keys=False),encoding='utf-8')
print('YAML listo:',DATA)

## 5. Auditar duplicados y grupos de procedencia
Verifica SHA-256 de imágenes y grupos del archivo de procedencia; no detecta todas las copias visuales ni escenas que fueron identificadas incorrectamente. Un cruce bloquea el entrenamiento.

In [ ]:
hashes=defaultdict(list); groups=defaultdict(set); origins=defaultdict(set); bases=defaultdict(set)
for split,imgs in IMAGES.items():
    for im in imgs:
        rec=CLEAN_META[im]
        hashes[sha_file(im)].append((split,im.name))
        groups[rec['grupo_escena']].add(split)
        origins[(rec['fuente'],rec['archivo_origen'])].add(split)
        base=Path(rec['archivo_origen']).stem.split('.rf.')[0]
        bases[(rec['fuente'],base)].add(split)
DUP=sum(len({s for s,_ in v})>1 for v in hashes.values())
LEAK={g:sorted(s) for g,s in groups.items() if len(s)>1}
ORIGIN_DUP=sum(len(s)>1 for s in origins.values())
BASE_DUP=sum(len(s)>1 for s in bases.values())
AUDIT={'hashes_cruzados':DUP,'hashes_repetidos_total':sum(len(v)>1 for v in hashes.values()),'grupos_cruzados':LEAK,'grupos_totales':len(groups),'origenes_cruzados':ORIGIN_DUP,'nombres_base_cruzados':BASE_DUP,'limitacion':'Grupos declarados; no garantiza independencia visual de todas las escenas.'}
(EVID/'auditoria_splits.json').write_text(json.dumps(AUDIT,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(AUDIT,indent=2,ensure_ascii=False))
assert not (DUP or LEAK or ORIGIN_DUP or BASE_DUP), 'Corrige cruces de datos antes de entrenar.'

## 6. Visualizar anotaciones y distribución
Muestras estratificadas por número de ejes y fuente, más casos marcados. Verde: v=2; naranja: v=1. Comprueba el centro de las ruedas, orden frente→atrás, ejes cercanos y cajas completas. Las reglas automáticas de orientación no garantizan que todas las etiquetas estén bien. Si encuentras errores, corrige las etiquetas y metadatos de origen y crea otro ZIP; no alteres el test para mejorar métricas.

In [ ]:
import matplotlib.pyplot as plt
rng=random.Random(SEED)
buckets=defaultdict(list)
for im in IMAGES['train']:
    rec=CLEAN_META[im]
    for m in rec['instancias']:
        buckets[(rec['fuente'],m['ejes_reales'])].append(im)
selected=[]
for key,imgs in sorted(buckets.items()):
    selected.append(rng.choice(list(dict.fromkeys(imgs))))
flagged=[im for im in IMAGES['train'] if any(m['requiere_revision'] for m in CLEAN_META[im]['instancias'])]
selected=list(dict.fromkeys(selected+rng.sample(flagged,min(4,len(flagged)))))
preview=EVID/'muestras_etiquetas'; preview.mkdir(exist_ok=True)
for j,im in enumerate(selected):
    pic=PILImage.open(im).convert('RGB'); pic.thumbnail((1280,1280)); w,h=pic.size; draw=ImageDraw.Draw(pic)
    for ((x,y,bw,bh),k),m in zip(GT[im],CLEAN_META[im]['instancias']):
        draw.rectangle(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h),outline='yellow',width=2)
        draw.text((max(0,(x-bw/2)*w),max(0,(y-bh/2)*h-12)),f"{m['ejes_reales']} ejes ref. {'REV' if m['requiere_revision'] else ''}",fill='yellow')
        for i,(kx,ky,v) in enumerate(k):
            if v>0:
                px,py=kx*w,ky*h; draw.ellipse((px-4,py-4,px+4,py+4),fill='lime' if v==2 else 'orange'); draw.text((px+5,py-10),f'e{i+1}',fill='yellow')
    pic.save(preview/f'{j:02d}.jpg'); plt.figure(figsize=(12,5)); plt.imshow(pic); plt.title(im.name); plt.axis('off'); plt.show(); plt.close()
fig,ax=plt.subplots(figsize=(9,4)); xs=list(range(1,8))
for i,s in enumerate(('train','valid','test')):
    counts=REPORT[s]['ejes_reales_referencia']; ax.bar([x+(i-1)*.25 for x in xs],[counts.get(x,0) for x in xs],width=.25,label=s)
ax.set_yscale('symlog',linthresh=1); ax.set_xticks(xs); ax.set_xlabel('Ejes de referencia'); ax.set_ylabel('Instancias (escala symlog)'); ax.legend(); fig.tight_layout(); fig.savefig(EVID/'distribucion_ejes.png',dpi=150); plt.show(); plt.close(fig)

## 7. Entrenar YOLOv8n-pose
100 épocas como límite, parada temprana 20, checkpoints cada cinco épocas. Se guardan en Drive. Si reinicias Colab, repite las secciones 1–5 con el mismo ZIP y configuración, cambia RESUME=True y ejecuta esta sección. Si el entrenamiento ya finalizó, pasa directamente a la sección 8. No reinicies mientras se guarda un checkpoint.

In [ ]:
LAST=RUN/'weights'/'last.pt'; BEST=RUN/'weights'/'best.pt'
RUN_META=RUN/'identidad_dataset.json'
identity={'zip_sha256':ZIP_HASH,'ultralytics':ultralytics.__version__,'kpt_shape':[7,3],'imgsz':IMGSZ,'epochs':EPOCHS,'batch':BATCH,'seed':SEED,'patience':PATIENCE,'allow_exclusions':ALLOW_EXCLUSIONS,'validacion':REPORT}
identity=json.loads(json.dumps(identity))
assert DATA.is_file() and not (DUP or LEAK or ORIGIN_DUP or BASE_DUP)
if RESUME:
    assert LAST.is_file() and RUN_META.is_file(), 'No hay checkpoint/identidad para reanudar.'
    assert json.loads(RUN_META.read_text())==identity, 'No cambies dataset, versión o configuración al reanudar.'
    model=YOLO(str(LAST))
    model.train(resume=True,data=str(DATA),device=0)
else:
    assert not RUN.exists() or not any(RUN.iterdir()), 'Ya existe esta ejecución: activa RESUME o usa otro RUN_NAME.'
    RUN.mkdir(parents=True,exist_ok=True)
    RUN_META.write_text(json.dumps(identity,indent=2),encoding='utf-8')
    model=YOLO('yolov8n-pose.pt')
    model.train(data=str(DATA),epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,device=0,workers=2,
                seed=SEED,patience=PATIENCE,fliplr=0.0,flipud=0.0,mosaic=0.0,close_mosaic=0,
                degrees=0.0,perspective=0.0,mixup=0.0,pretrained=True,plots=True,save=True,save_period=5,
                project=str(BASE/'entrenamientos'),name=RUN_NAME,exist_ok=True)
assert BEST.is_file(), f'No se encontró {BEST}'
print('Mejor checkpoint:', BEST)

## 8. Evaluar el mejor checkpoint: métricas de cajas y pose
Usa test solo como evaluación final. La precisión y mAP de pose de Ultralytics no son exactitud del número de ejes; son métricas de localización. La métrica OKS usa el esquema personalizado de siete puntos de la biblioteca.

In [ ]:
BEST=RUN/'weights'/'best.pt'
assert BEST.is_file(), 'Primero entrena o recupera best.pt.'
model=YOLO(str(BEST))
assert model.task=='pose' and model.names=={0:'vehiculo'}
assert list(model.model.yaml['kpt_shape'])==[7,3], 'El checkpoint no corresponde a M3.'
metrics=model.val(data=str(DATA),split='test',device=0,imgsz=IMGSZ,batch=BATCH,plots=True,project=str(EVID),name='test_pose',exist_ok=True)
POSE_METRICS={str(k):float(v) for k,v in metrics.results_dict.items()}
(EVID/'metricas_pose_test.json').write_text(json.dumps(POSE_METRICS,indent=2),encoding='utf-8')
print(json.dumps(POSE_METRICS,indent=2))

## 9. Evaluar conteo de ejes y guardar evidencias
Las cajas predichas se asocian uno a uno con cajas anotadas mediante IoU ≥ 0.50, de mayor a menor IoU. Se cuentan puntos con confianza ≥ 0.50 (sin asumir que un punto oculto exista o no). Se comparan contra dos referencias: puntos anotados (v>0) y número de ejes reportado. También se reportan puntos explícitamente visibles (v=2) y un subconjunto de referencias completas no marcadas para revisión. El error medio absoluto (MAE) se calcula solo en vehículos asociados; la exactitud global penaliza los vehículos no detectados. Se registran predicciones sin correspondencia, conteos por número de ejes y fuente, y resultados por instancia. Las exclusiones previas se documentan: no se consideran aciertos y no forman parte del denominador de evaluación de etiquetas válidas. Las referencias USP se basan en las anotaciones de Axle y no son una verificación independiente del total físico. No ajustar umbrales con test.

In [ ]:
import csv

def iou(a,b):
    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union>0 else 0.0

def xyxy(b):
    x,y,w,h=b; return (x-w/2,y-h/2,x+w/2,y+h/2)

def match_boxes(gt,pred,threshold=.5):
    cand=[]
    for gi,g in enumerate(gt):
        for pi,b in enumerate(pred):
            score=iou(g,b)
            if score>=threshold: cand.append((score,gi,pi))
    matches={}; used=set()
    for score,gi,pi in sorted(cand,reverse=True):
        if gi not in matches and pi not in used: matches[gi]=(pi,score); used.add(pi)
    return matches,used

def summarize(records,field):
    n=len(records); matched=[r for r in records if r['predicho'] is not None]; m=len(matched)
    correct=sum(r[field]==r['predicho'] for r in matched)
    return {'total':n,'detectados':m,'correctos':correct,'exactitud_incluye_no_detectados':correct/n if n else None,'exactitud_entre_detectados':correct/m if m else None,'mae_detectados':sum(abs(r[field]-r['predicho']) for r in matched)/m if m else None}

def infer_counts(res):
    boxes=res.boxes.xyxyn.cpu().numpy() if res.boxes is not None else np.empty((0,4))
    if not len(boxes): return boxes,[]
    if res.keypoints is None or res.keypoints.conf is None: raise RuntimeError('El modelo no devuelve confianza por keypoint')
    conf=res.keypoints.conf.cpu().numpy(); assert conf.shape==(len(boxes),7)
    return boxes,(conf>=KPT_CONF).sum(axis=1).astype(int).tolist()

RESULT_ROWS=[]; FP=0; PRED_TOTAL=0
for start in range(0,len(IMAGES['test']),BATCH):
    chunk=IMAGES['test'][start:start+BATCH]
    predictions=model.predict(source=[str(p) for p in chunk],imgsz=IMGSZ,conf=BOX_CONF,device=0,max_det=MAX_DET,verbose=False)
    assert len(predictions)==len(chunk), 'Predicciones incompletas; vuelve a ejecutar la evaluación.'
    for im,res in zip(chunk,predictions):
        rows=GT[im]; rec=CLEAN_META[im]; boxes,counts=infer_counts(res)
        matches,used=match_boxes([xyxy(b) for b,k in rows],boxes)
        FP+=len(boxes)-len(used); PRED_TOTAL+=len(boxes)
        for gi,((b,k),m) in enumerate(zip(rows,rec['instancias'])):
            pi,score=matches.get(gi,(None,None))
            RESULT_ROWS.append({'imagen':im.name,'fuente':rec['fuente'],'grupo':rec['grupo_escena'],'instancia':gi,'real':m['ejes_reales'],'anotados':int((k[:,2]>0).sum()),'visibles':int((k[:,2]==2).sum()),'revision':m['requiere_revision'],'predicho':counts[pi] if pi is not None else None,'iou':score})
assert RESULT_ROWS, 'No hay referencias de prueba.'
assert len(RESULT_ROWS)==REPORT['test']['instancias']
full=[r for r in RESULT_ROWS if r['real']==r['visibles'] and not r['revision']]
matched=sum(r['predicho'] is not None for r in RESULT_ROWS)
confusion=Counter((r['real'],str(r['predicho']) if r['predicho'] is not None else 'no_detectado') for r in RESULT_ROWS)
COUNT_METRICS={'referencia':'metadatos_procedencia.json y etiquetas YOLO v2','umbral_keypoint':KPT_CONF,'umbral_bbox':BOX_CONF,'iou_match':0.5,'todas':summarize(RESULT_ROWS,'real'),'puntos_anotados':summarize(RESULT_ROWS,'anotados'),'puntos_visibles_v2':summarize(RESULT_ROWS,'visibles'),'todos_los_ejes_visibles_sin_revision':summarize(full,'real'),'por_ejes':{str(i):summarize([r for r in RESULT_ROWS if r['real']==i],'real') for i in range(1,8)},'por_fuente':{s:summarize([r for r in RESULT_ROWS if r['fuente']==s],'real') for s in sorted({r['fuente'] for r in RESULT_ROWS})},'predicciones_sin_match':FP,'predicciones_totales':PRED_TOTAL,'precision_deteccion_iou50':matched/PRED_TOTAL if PRED_TOTAL else None,'recall_deteccion_iou50':matched/len(RESULT_ROWS),'imagenes_test_utilizadas':len(IMAGES['test']),'imagenes_test_excluidas':sum(r['split']=='test' for r in REJECTED),'metadatos_test_sin_imagen':sum(s=='test' for s,n in MISSING_META),'confusion':[{'real':a,'predicho':b,'n':v} for (a,b),v in sorted(confusion.items())],'nota':'Referencia anotada, no certificación del número físico de ejes; no detectados penalizados en exactitud pero excluidos del MAE.'}
(EVID/'metricas_conteo_ejes.json').write_text(json.dumps(COUNT_METRICS,indent=2,ensure_ascii=False),encoding='utf-8')
with (EVID/'conteo_por_instancia.csv').open('w',encoding='utf-8-sig',newline='') as f:
    wr=csv.DictWriter(f,fieldnames=list(RESULT_ROWS[0])); wr.writeheader(); wr.writerows(RESULT_ROWS)
print(json.dumps(COUNT_METRICS,indent=2,ensure_ascii=False))

## 10. Visualizar matriz de conteo de ejes
La última columna contiene vehículos de referencia sin detección asociada. Las predicciones sin correspondencia se reportan por separado.

In [ ]:
cols=[str(i) for i in range(8)]+['ND']; matrix=np.zeros((7,len(cols)),dtype=int)
for (real,pred),n in confusion.items(): matrix[real-1,cols.index('ND' if pred=='no_detectado' else pred)]=n
fig,ax=plt.subplots(figsize=(10,6)); heat=ax.imshow(matrix,cmap='Blues'); fig.colorbar(heat,ax=ax)
ax.set_xticks(range(len(cols)),cols); ax.set_yticks(range(7),range(1,8)); ax.set_xlabel('Puntos predichos'); ax.set_ylabel('Ejes de referencia')
for i in range(7):
    for j in range(len(cols)): ax.text(j,i,str(matrix[i,j]),ha='center',va='center',color='white' if matrix[i,j]>matrix.max()/2 else 'black')
fig.tight_layout(); fig.savefig(EVID/'matriz_conteo_ejes.png',dpi=160); plt.show(); plt.close(fig)

## 11. Exportar pesos y manifiesto para Antigravity
Solo después de evaluar. Se respalda el modelo anterior si existe y se conserva una copia versionada del nuevo modelo en Drive. Los umbrales son iniciales, no autorización de cobro.

In [ ]:
from datetime import datetime, timezone
DEST=BASE/'modelos'/'ejes.pt'; MANIFEST=BASE/'modelos'/'ejes.manifest.json'
if DEST.exists() and sha_file(DEST)!=sha_file(BEST):
    old=BASE/'modelos'/'historial'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    old.mkdir(parents=True); shutil.copy2(DEST,old/DEST.name)
    if MANIFEST.exists(): shutil.copy2(MANIFEST,old/MANIFEST.name)
shutil.copy2(BEST,DEST)
manifest={'modelo':'M3','archivo':DEST.name,'sha256':sha_file(DEST),'task':'pose','arquitectura':'YOLOv8n-pose','names':{'0':'vehiculo'},'kpt_shape':[7,3],'keypoints':KEYPOINT_NAMES,'kpt_names':{'0':KEYPOINT_NAMES},'orden':'delantero_a_trasero','imgsz':IMGSZ,'conf_bbox':BOX_CONF,'conf_keypoint':KPT_CONF,'dataset_zip_sha256':ZIP_HASH,'dataset':'Vehicle Wheel Pose + Truck Image Dataset v2.0.0 (USP), adaptados por Antigravity','dataset_version':'v2','ultralytics':ultralytics.__version__,'torch':torch.__version__,'run':RUN_NAME,'fecha_utc':datetime.now(timezone.utc).isoformat(),'metricas_pose_test':POSE_METRICS,'metricas_conteo_ejes':COUNT_METRICS,'auditoria_splits':AUDIT,'requiere_revision_si_incierto':True,'validacion_produccion':False,'metodo_conteo':'keypoints con confianza >= conf_keypoint; requiere comprobación de visibilidad y asociación','max_ejes':7,'fuente_complementaria':META['fuente_complementaria'],'reporte_dataset':REPORT}
MANIFEST.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Archivos listos:',DEST,MANIFEST,sep='\n')
versioned=BASE/'modelos'/'M3_v2'; versioned.mkdir(exist_ok=True)
shutil.copy2(DEST,versioned/DEST.name); shutil.copy2(MANIFEST,versioned/MANIFEST.name)

## 12. Evidencias visuales de predicción y prueba opcional
Cada caja muestra el conteo estimado mediante el mismo umbral de la evaluación. En casos incompletos el conteo puede subestimar los ejes físicos.

In [ ]:
from IPython.display import display

def render_prediction(im,res):
    pic=PILImage.open(im).convert('RGB'); w,h=pic.size; draw=ImageDraw.Draw(pic)
    boxes,counts=infer_counts(res)
    if len(boxes):
        pts=res.keypoints.xyn.cpu().numpy(); conf=res.keypoints.conf.cpu().numpy()
        for (x1,y1,x2,y2),ks,cs,n in zip(boxes,pts,conf,counts):
            draw.rectangle((x1*w,y1*h,x2*w,y2*h),outline='lime',width=max(2,w//700))
            draw.text((x1*w,max(0,y1*h-14)),f'{n} puntos estimados',fill='yellow')
            for j,((x,y),c) in enumerate(zip(ks,cs)):
                if c>=KPT_CONF:
                    px,py=x*w,y*h; r=max(4,w//300); draw.ellipse((px-r,py-r,px+r,py+r),fill='yellow'); draw.text((px+r,py-r),f'e{j+1}',fill='lime')
    return pic
pred_dir=EVID/'predicciones'; pred_dir.mkdir(exist_ok=True)
rng=random.Random(SEED); chosen=[]
for n in range(2,8):
    opts=[im for im in IMAGES['test'] if any(m['ejes_reales']==n for m in CLEAN_META[im]['instancias'])]
    if opts: chosen.append(rng.choice(opts))
for im in dict.fromkeys(chosen):
    res=model.predict(str(im),imgsz=IMGSZ,conf=BOX_CONF,device=0,max_det=MAX_DET,verbose=False)[0]
    pic=render_prediction(im,res); pic.thumbnail((1280,1280)); pic.save(pred_dir/(im.stem+'.jpg')); display(pic)
for f in [RUN/'results.png',RUN/'confusion_matrix.png',RUN/'confusion_matrix_normalized.png']:
    if f.exists(): display(PILImage.open(f))

In [ ]:
PROBAR_IMAGEN_PROPIA=False  # Cambia a True para subir una imagen de prueba autorizada.
if PROBAR_IMAGEN_PROPIA:
    from google.colab import files
    for filename in files.upload():
        res=model.predict(filename,imgsz=IMGSZ,conf=BOX_CONF,device=0,max_det=MAX_DET,verbose=False)[0]
        pic=render_prediction(filename,res); pic.thumbnail((1280,1280)); display(pic)
        print('Puntos detectados por vehículo:', infer_counts(res)[1])

## 13. Entrega y revisión final
- Copia `Sistema-Peaje/modelos/ejes.pt` y `ejes.manifest.json` a `IA PEAJE/inferencia/modelos/` para Antigravity. No reemplaces `vehiculos.pt` de M2.
- Conserva `entrenamientos/M3_keypoints_v2/weights/best.pt`, `last.pt`, `results.csv`, `args.yaml`, `results.png` y las evidencias de `evidencias/M3_keypoints_v2/`.
- Revisa la exactitud por ejes (especialmente 4, 6 y 7), MAE entre detectados, casos sin detección y predicciones falsas. Evalúa con vehículos propios independientes y con condiciones de la cámara real. No conviertas los ejemplos de test en datos de entrenamiento sin establecer una nueva prueba independiente.
- M3 no confirma por sí solo la tarifa ni el total físico cuando los ejes están ocultos o exceden siete; el sistema debe manejar incertidumbre y revisión.

**Fuentes:** Vehicle Wheel Pose (Roboflow, dataset original del proyecto); Marcomini, L. A., & Cunha, A. L. (2023). Truck Image Dataset v2.0.0, Zenodo, https://zenodo.org/records/7695824 (CC BY 4.0 según el reporte de conversión). Conversión y metadatos proporcionados por Antigravity.